# 0.12007 | 19th / 3,462 | Silver Medal | March Machine Learning Mania 2026

This was my first Kaggle competition (and my first real ML project). I'm a data engineer by trade, so I came at this from the "how do I build a reliable pipeline" angle rather than deep ML expertise. I built the model iteratively across 12 levels over about a week, adding one idea at a time and measuring the impact of each.

The model correctly predicted **Michigan as national champion** and went **87/104 (83.7%)** across all tournament games.

Full code is on [GitHub](https://github.com/jswansonco/march-machine-learning-mania).

## Approach

The core idea: **predict point differential, not win/loss.** A 20-point blowout and a 1-point squeaker are both "wins" in binary classification, but they carry very different information about team strength. Regressing on margin gives the model a much richer training signal. I then recovered win probabilities with a calibration step.

Men's and women's tournaments were treated as a single dataset with a binary gender indicator. All tournament games from 2003-2025 were used for training (~1,300 games, doubled to ~2,600 via symmetric augmentation).

## Feature Engineering (53 features)

Every game in training was doubled symmetrically (Team A vs B and Team B vs A) with overtime stats normalized to 40-minute equivalents. This lets the model learn from both perspectives without directional bias.

| Group | # | What | Why it helps |
|-------|---|------|-------------|
| Seeds | 3 | Seed for each team + diff | Strongest single predictor (0.1671 Brier alone) |
| Box scores | 18 | Season-avg points, rebounds, blocks, fouls (team + opponent) | Baseline team quality signal |
| Late-season form | 2 | Avg point diff in final 2 weeks | Catches teams peaking/slumping into March |
| Elo | 3 | Margin-weighted Elo per team + diff | Dynamic strength with season carry-over |
| GLM quality | 2 | Bradley-Terry quality scores | Principled team strength from point diffs |
| Massey Ordinals | 4 | KenPom rank + avg of ~60 systems | Consensus rankings are very predictive |
| BartTorvik | 18 | AdjOE, AdjDE, Barthag, WAB, SOS, Tempo + diffs | **Single biggest improvement (+3.1%)** |
| Gender | 1 | Men's vs women's flag | 4th most important feature — different upset dynamics |

### Why BartTorvik Was the Breakthrough

Before Level 12, I was computing team strength from raw box scores — season-average points, rebounds, field goal attempts, etc. The problem is that raw box scores don't account for pace or opponent quality. A team that scores 85 points per game against a weak schedule in a fast conference looks the same as one that scores 85 against elite competition. You can try to correct for this with strength-of-schedule adjustments, but you're basically rebuilding what analytics sites like BartTorvik already do better.

[BartTorvik's T-Rank](https://www.kaggle.com/datasets/jswanson/barttorvik-t-rank-2015-2026) provides tempo-adjusted, opponent-adjusted efficiency metrics. The key columns:

- **AdjOE / AdjDE** — Offensive and defensive efficiency (points per 100 possessions, adjusted for opponent strength). This is the gold standard for measuring how good a team actually is on each end of the floor, stripped of pace and schedule effects.
- **Barthag** — An overall power rating representing a team's expected win probability against an average D1 team. This ended up as the **5th most important feature** in the model.
- **WAB (Wins Above Bubble)** — How many more wins a team has than a bubble-quality team would expect against the same schedule. This captures "are they a legitimate tournament team or did they just beat up on a weak conference?"
- **Tempo** — Adjusted pace of play (possessions per 40 minutes). Fast teams and slow teams create fundamentally different game environments.

The hypothesis was that these pre-computed, expert-quality metrics would give XGBoost strictly better signal than anything I could derive from raw box scores — and that turned out to be right. Adding Torvik features dropped the LOSO Brier from 0.1650 to 0.1599 (3.1% improvement), the single biggest jump in the entire model evolution. Once Torvik was in the mix, the box score Four Factors (eFG%, turnover rate, offensive rebound rate, free throw rate) became completely redundant.

The key decision was feeding these as **training features for XGBoost** rather than building a separate Torvik-based model and ensembling. As direct features, the trees can learn interactions — like "a team with high AdjDE but a low seed is being underestimated" or "Barthag differences matter more in the women's tournament." You lose those interactions if you just average two models together.

### Elo System

I used a margin-weighted Elo with 60% season-to-season carry-over. The update scales by `log(1 + |margin|)`, giving more credit to blowouts:

In [ ]:
def compute_better_elo(regular_data):
    K, WIDTH, MARGIN_FACTOR, REVERSION, INIT = 32, 400, 0.8, 0.4, 1500
    ratings, snapshots, current_season = {}, {}, None
    for season in sorted(regular_data["Season"].unique()):
        if current_season is not None:
            for team in ratings:
                ratings[team] = INIT + (ratings[team] - INIT) * (1 - REVERSION)
        current_season = season
        ss = regular_data[(regular_data["Season"] == season) & (regular_data["win"] == 1)].sort_values("DayNum")
        for _, row in ss.iterrows():
            w, l = int(row["T1_TeamID"]), int(row["T2_TeamID"])
            if w not in ratings: ratings[w] = INIT
            if l not in ratings: ratings[l] = INIT
            exp_w = 1.0 / (1.0 + 10.0 ** ((ratings[l] - ratings[w]) / WIDTH))
            margin = row["T1_Score"] - row["T2_Score"]
            update = K * np.log(1 + abs(margin)) * MARGIN_FACTOR * (1 - exp_w)
            ratings[w] += update
            ratings[l] -= update
        for team, rating in ratings.items():
            snapshots[(season, team)] = rating
    return snapshots

### GLM Team Quality

A Gaussian GLM fit per season on point differential with team fixed effects — basically Bradley-Terry. This gives every tournament team a latent quality score grounded in a principled statistical model.

In [ ]:
# Fit per season, per gender
glm = sm.GLM.from_formula(
    "PointDiff ~ -1 + T1_TeamID + T2_TeamID",
    data=subset,
    family=sm.families.Gaussian()
).fit()
# Extract T1 coefficients as team quality scores
quality_scores = glm.params[glm.params.index.str.startswith("T1_")]

## Modeling

XGBoost regression on point differential. The hyperparameters lean heavily on regularization — with only ~2,600 training rows (tournament games are scarce), overfitting is the main risk.

In [ ]:
param = {
    "objective": "reg:squarederror",
    "booster": "gbtree",
    "eta": 0.0093,           # low LR + many rounds for smooth predictions
    "subsample": 0.6,
    "colsample_bynode": 0.8,
    "num_parallel_tree": 2,  # mini random forest per boosting round
    "min_child_weight": 4,
    "max_depth": 4,
    "tree_method": "hist",
    "grow_policy": "lossguide",
    "max_bin": 38,
}
num_rounds = 704

### Validation: Leave-One-Season-Out (LOSO)

Each of the 22 tournament seasons (2003-2025, minus 2020) gets held out in turn. The model trains on the other 21 seasons and predicts the held-out tournament. This respects temporal structure and avoids leakage.

Crucially, **all 22 models are kept for inference.** For each 2026 matchup, every model makes a prediction and they're averaged. This is essentially a free ensemble that reduces variance.

In [ ]:
models = {}
for oof_season in seasons:
    X_tr = tourney_data.loc[tourney_data["Season"] != oof_season, features].values
    y_tr = tourney_data.loc[tourney_data["Season"] != oof_season, "PointDiff"].values
    dtrain = DMatrix(X_tr, label=y_tr, feature_names=features)
    models[oof_season] = xgb_train(params=param, dtrain=dtrain, num_boost_round=num_rounds)

## Probability Calibration

XGBoost outputs point differentials, not probabilities. To convert them, I fit a quintic (degree-5) spline on the out-of-fold margin predictions against binary win indicators. The spline captures the natural S-curve while staying smooth — the relationship between margin and win probability isn't perfectly logistic, and the spline handles the tails gracefully.

In [ ]:
CLIP_DIFF = 25
dat = sorted(zip(oof_preds, [int(t > 0) for t in oof_targets]), key=lambda x: x[0])
pred_sorted, label_sorted = zip(*dat)
spline_model = UnivariateSpline(
    np.clip(pred_sorted, -CLIP_DIFF, CLIP_DIFF),
    label_sorted,
    k=5  # quintic
)
# At inference: average margins across 22 models, then apply spline
probs = np.clip(spline_model(np.clip(avg_margins, -CLIP_DIFF, CLIP_DIFF)), 0.01, 0.99)

### Final Blend

The submission blended two models: **80% Level 12** (full feature set with BartTorvik) and **20% Level 9** (same architecture but without BartTorvik — just box scores, Elo, GLM, Massey). The simpler model acts as a regularizer that pulls extreme predictions back toward center.

In [ ]:
BLEND_WEIGHT = 0.8  # 80% Level 12, 20% Level 9
final_pred = BLEND_WEIGHT * level12_pred + (1 - BLEND_WEIGHT) * level9_pred
final_pred = final_pred.clip(0.01, 0.99)

## Feature Importance

XGBoost tracks how much each feature improves predictions across all trees — called "total gain." Higher gain means the feature was more useful for reducing prediction error. The top 10:

| Rank | Feature | Source | What it tells us |
|------|---------|--------|-----------------|
| 1 | Seed_diff | Seeds | Seed gap between teams — by far the strongest signal |
| 2 | elo2_diff | Elo | Elo rating gap — nearly as important as seeds |
| 3 | T2/T1_seed | Seeds | Individual team seeds (not just the diff) |
| 4 | men_women | Gender | Men's vs women's — tournaments behave differently |
| 5 | tv_barthag_diff | BartTorvik | Power rating gap — the top Torvik feature |
| 6 | T1/T2_quality | GLM | Bradley-Terry quality gap |
| 7 | POM_diff | KenPom | KenPom ranking gap |
| 8 | T2_POM | KenPom | Opponent's KenPom rank on its own |
| 9 | MasseyAvg_diff | Massey | Average across ~60 ranking systems |
| 10 | tv_adjoe_diff | BartTorvik | Offensive efficiency gap |

The takeaway: seeds and Elo do the heavy lifting, but the model gets meaningful lift from having multiple independent views of team quality (Torvik, GLM, KenPom). Each captures something slightly different, and XGBoost learns when to trust which signal.

## Tournament Results

**87/104 games correct (83.7%), Final Brier: 0.12007**

| Split | Record | Brier |
|-------|--------|-------|
| Men's overall | 52/63 (82.5%) | 0.1428 |
| Women's overall | 35/41 (85.4%) | 0.1030 |

**Highlights:**
- Bracket predicted Michigan as champion — **correct**
- Women's R32: 9/9 perfect, Women's E8: 4/4 with 0.011 Brier
- Men's Final Four: 3/3 (model had Michigan at 81.7% in the championship)

**Biggest misses:** (11) Texas over (3) Gonzaga (model gave Texas 9.3%, Brier 0.82), and UConn 77.9% over South Carolina in the Women's Final Four (SC won 62-48, ending UConn's 38-0 season).

## What Worked

1. **BartTorvik as training features** — 3.1% Brier improvement, the single biggest gain
2. **Point-diff regression** over classification — richer signal, better calibration
3. **Margin-weighted Elo** with carry-over — captures dynamic team strength
4. **Massey Ordinals** — consensus of ~60 ranking systems is hard to beat
5. **Quintic spline calibration** — smooth, non-parametric margin-to-probability
6. **LOSO ensemble** of 22 models — free variance reduction
7. **80/20 blend** — regularizes extreme predictions

## What Didn't Work

- **Conference strength** — redundant with Elo and Massey
- **Box score Four Factors** (eFG%, TO%, ORB%, FTR) — redundant once BartTorvik is in the mix
- **Probability clipping** — spline already produced well-calibrated outputs
- **Post-hoc pace compression** (Level 13 experiment) — too blunt as a linear adjustment
- **Clutch factor** (Level 13) — low importance; probably needs play-by-play data

## External Data

- **[BartTorvik T-Rank](https://www.kaggle.com/datasets/jswanson/barttorvik-t-rank-2015-2026)** (barttorvik.com) — men's 2015-2026, women's 2021-2026. Scraped and uploaded as a Kaggle dataset.
- **Massey Ordinals** — included in competition data
- Everything else derived from competition-provided box scores and results

## Tools

Python, XGBoost, pandas, statsmodels, scipy. Built iteratively with [Claude Code](https://claude.ai/code) as a coding assistant.